#### Importing Required Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import joblib

from sklearn.inspection import permutation_importance

#### Loading the Saved Model and Preprocessor

In [4]:
# Loading the final trained stacking regression model.

final_model = joblib.load(
    '../models/final_stacking_model.pkl'
)

# Loading the fitted preprocessing pipeline.

preprocessor = joblib.load(
    '../models/preprocessor.pkl'
)

#### Load the Feature-Engineered Data

In [5]:
# Loading the feature-engineered dataset used for model development.

df = pd.read_csv(
    '../data/processed/engineered_data.csv'
)

df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Is_Weekend,Inventory_to_Sales_Ratio,Inventory_Gap,Price_Difference,Price_Difference_Percentage,Promotion_Discount,Previous_Demand,Previous_Units_Sold,Rolling_7_Day_Demand,Rolling_7_Day_Sales
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,1,1.911765,93,-13.01,-15.175551,0,NaN,NaN,NaN,NaN
1,2022-01-02,S001,P0001,Electronics,North,93,71,0,65.63,5,...,1,1.309859,22,-8.03,-10.901439,0,115.0,102.0,NaN,NaN
2,2022-01-03,S001,P0001,Electronics,North,274,142,229,68.55,15,...,0,1.929577,132,-12.18,-15.087328,15,84.0,71.0,NaN,NaN
3,2022-01-04,S001,P0001,Electronics,North,132,42,0,61.66,10,...,0,3.142857,90,6.78,12.354227,0,132.0,142.0,NaN,NaN
4,2022-01-05,S001,P0001,Electronics,North,319,129,0,59.56,25,...,0,2.472868,190,2.22,3.871643,25,67.0,42.0,NaN,NaN


In [6]:
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (76000, 31)

Columns:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality', 'Epidemic', 'Demand', 'Year', 'Month', 'Week', 'Day', 'Day_of_Week', 'Is_Weekend', 'Inventory_to_Sales_Ratio', 'Inventory_Gap', 'Price_Difference', 'Price_Difference_Percentage', 'Promotion_Discount', 'Previous_Demand', 'Previous_Units_Sold', 'Rolling_7_Day_Demand', 'Rolling_7_Day_Sales']


#### Separate Features and Target

In [7]:
# Separating the target variable from the input features.

X = df.drop(
    columns=['Demand']
)

y = df['Demand']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (76000, 30)
Target Shape: (76000,)


In [8]:
# Recreating the same train-test split used during model development.

split_index = 63800

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

print("Training Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

Training Shape: (63800, 30)
Testing Shape: (12200, 30)
Training Target Shape: (63800,)
Testing Target Shape: (12200,)


#### Transform the Test Data

In [9]:
# Transforming the test features using the fitted preprocessor.

X_test_processed = preprocessor.transform(
    X_test
)

print("Processed Testing Shape:", X_test_processed.shape)

Processed Testing Shape: (12200, 64)


#### Generating Predictions

In [10]:
# Generating predictions from the final stacking model.

y_pred = final_model.predict(
    X_test_processed
)

print("Number of Predictions:", len(y_pred))

Number of Predictions: 12200


#### Permutation Feature Importance

For each feature, the method:

Takes the test data.
Randomly shuffles one feature.
Makes predictions again.
Checks how much the model's performance gets worse.

If shuffling a feature causes a big performance drop:

That feature was important.

If almost nothing changes:

That feature probably wasn't contributing much to the predictions.

##### Calculate Permutation Importance

In [11]:
'''# Calculating permutation importance using the test dataset.
# A small number of repeats is used to keep computation manageable.

permutation_result = permutation_importance(
    final_model,
    X_test_processed,
    y_test,
    scoring='neg_mean_absolute_error',
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

print("Permutation importance calculated.")'''

'# Calculating permutation importance using the test dataset.\n# A small number of repeats is used to keep computation manageable.\n\npermutation_result = permutation_importance(\n    final_model,\n    X_test_processed,\n    y_test,\n    scoring=\'neg_mean_absolute_error\',\n    n_repeats=3,\n    random_state=42,\n    n_jobs=-1\n)\n\nprint("Permutation importance calculated.")'

In [12]:
# Creating a small representative sample for explainability.

X_explain = X_test_processed[:500]
y_explain = y_test.iloc[:500]

print("Explainability Sample Shape:", X_explain.shape)
print("Explainability Target Shape:", y_explain.shape)

Explainability Sample Shape: (500, 64)
Explainability Target Shape: (500,)


##### Lightweight Permutation Importance

In [13]:
# Calculating lightweight permutation importance.

permutation_result = permutation_importance(
    final_model,
    X_explain,
    y_explain,
    scoring='neg_mean_absolute_error',
    n_repeats=1,
    random_state=42,
    n_jobs=1
)

print("Permutation importance calculated successfully.")

Permutation importance calculated successfully.


##### Getting the Processed Feature Names

In [14]:
# Extracting the feature names generated by the preprocessing pipeline.

feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print("\nFirst 20 features:")
print(feature_names[:20])

Number of processed features: 64

First 20 features:
['categorical__Store ID_S001' 'categorical__Store ID_S002'
 'categorical__Store ID_S003' 'categorical__Store ID_S004'
 'categorical__Store ID_S005' 'categorical__Product ID_P0001'
 'categorical__Product ID_P0002' 'categorical__Product ID_P0003'
 'categorical__Product ID_P0004' 'categorical__Product ID_P0005'
 'categorical__Product ID_P0006' 'categorical__Product ID_P0007'
 'categorical__Product ID_P0008' 'categorical__Product ID_P0009'
 'categorical__Product ID_P0010' 'categorical__Product ID_P0011'
 'categorical__Product ID_P0012' 'categorical__Product ID_P0013'
 'categorical__Product ID_P0014' 'categorical__Product ID_P0015']


##### Building the Proper Importance Table

In [15]:
# Creating a feature importance table with meaningful processed feature names.

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': permutation_result.importances_mean
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
).reset_index(drop=True)

importance_df.head(15)

,Feature,Importance
0,numerical__Inventory_to_Sales_Ratio,34.216962
1,numerical__Inventory Level,22.522068
2,numerical__Units Ordered,11.245528
3,numerical__Promotion,3.468985
4,numerical__Inventory_Gap,2.406649
5,numerical__Epidemic,2.400952
6,categorical__Seasonality_Summer,2.384696
7,categorical__Weather Condition_Sunny,2.328203
8,numerical__Rolling_7_Day_Demand,1.816906
9,numerical__Previous_Demand,1.081752


The big finding

Inventory_to_Sales_Ratio is far ahead of everything else.

Inventory_to_Sales_Ratio   34.22
Inventory Level            22.52
Units Ordered              11.25
Promotion                    3.47
...

That means the model's predictions are particularly sensitive to the information contained in the inventory-to-sales ratio.

But I don't want us to immediately declare that this is the most important business driver of Demand.

Why?

Because we need to verify how that feature was constructed.

If Inventory_to_Sales_Ratio uses information that would only be known after or at the same time as the Demand we're trying to predict, it could introduce target leakage or make the feature unrealistic for forecasting.

That's especially important for a project like ours.

##### Inspecting the Feature Engineering